# PAISA optional OGDC LSTM experiment

This is an optional Colab workflow, separate from the required CPU M-001 through M-004 pipeline. It reuses PAISA's target, feature, and chronological-split preparation, then applies a small Keras LSTM. TensorFlow is installed only in this notebook and is not a default project dependency.

The output is an academic experiment and is not financial advice.

## 1. Clone or use PAISA and install optional TensorFlow

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Arfa01/PAISA.git"
GIT_REF = os.environ.get("PAISA_GIT_REF", "main")
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_DIR = Path("/content/PAISA")
else:
    current = Path.cwd()
    REPO_DIR = current if (current / "pyproject.toml").exists() else current / "PAISA"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / "pyproject.toml").exists():
    raise RuntimeError(f"Existing path is not a PAISA checkout: {REPO_DIR}")

os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow>=2.16,<3.0"], check=True)
print("Repository:", REPO_DIR)
print("TensorFlow is notebook-local and is not required by the CPU trainer.")

## 2. Fetch or load OGDC and call shared preparation

Set `PAISA_PRICES_PATH` to a cleaned real-data CSV to avoid a live request. Otherwise the existing PAISA PSX pipeline is used. The reusable helper owns targets, technical features, feature ordering, and chronological split dates.

In [ ]:
from datetime import date

import pandas as pd

from paisa.experiments.trainer import prepare_training_data
from paisa.pipeline import run_v0_pipeline

prices_value = os.environ.get("PAISA_PRICES_PATH")
if prices_value:
    prices_path = Path(prices_value)
else:
    run_v0_pipeline(
        symbols=["OGDC"],
        provider_name="psx-dps",
        start=date(2021, 1, 1),
        train=False,
    )
    prices_path = Path("data/processed/psx_prices.csv")

prices = pd.read_csv(prices_path)
prepared = prepare_training_data(prices, symbol="OGDC")
print("Dataset SHA-256:", prepared.dataset_sha256)
print("Ordered features:", list(prepared.feature_columns))
print({name: len(getattr(prepared.splits, name)) for name in ("train", "validation", "test")})

## 3. Build leakage-safe sequences

The scaler is fit on training rows only. Sequences are formed independently inside each chronological split, and Keras training is not shuffled.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from paisa.experiments.experiment_config import TARGET_NAME

SEQUENCE_LENGTH = 20
feature_columns = list(prepared.feature_columns)
scaler = StandardScaler()
train_values = scaler.fit_transform(prepared.splits.train[feature_columns])
validation_values = scaler.transform(prepared.splits.validation[feature_columns])
test_values = scaler.transform(prepared.splits.test[feature_columns])

def make_sequences(frame, values):
    if len(frame) < SEQUENCE_LENGTH:
        raise ValueError(f"Split has {len(frame)} rows; need at least {SEQUENCE_LENGTH}")
    end_indexes = range(SEQUENCE_LENGTH - 1, len(frame))
    X = np.asarray([values[i - SEQUENCE_LENGTH + 1:i + 1] for i in end_indexes], dtype=np.float32)
    rows = frame.iloc[list(end_indexes)].reset_index(drop=True)
    y = rows[TARGET_NAME].astype(int).to_numpy(dtype=np.int32)
    return X, y, rows

X_train, y_train, train_rows = make_sequences(prepared.splits.train, train_values)
X_validation, y_validation, validation_rows = make_sequences(prepared.splits.validation, validation_values)
X_test, y_test, test_rows = make_sequences(prepared.splits.test, test_values)
print(X_train.shape, X_validation.shape, X_test.shape)

## 4. Train the optional LSTM

In [ ]:
import tensorflow as tf

RANDOM_SEED = 42
tf.keras.utils.set_random_seed(RANDOM_SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(SEQUENCE_LENGTH, len(feature_columns))),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
history = model.fit(
    X_train, y_train,
    validation_data=(X_validation, y_validation),
    epochs=50,
    batch_size=32,
    shuffle=False,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True)],
    verbose=1,
)

## 5. Evaluate and save contract-compatible test predictions

In [ ]:
import json
from datetime import datetime, timezone

from IPython.display import display
from paisa.experiments.artifacts import make_zip_from_directory, write_json
from paisa.experiments.evaluation import binary_classification_metrics

probability_up = model.predict(X_test, verbose=0).reshape(-1)
predicted_class = (probability_up > 0.5).astype(int)
metrics = binary_classification_metrics(y_test, predicted_class, probability_up)
display(pd.DataFrame([metrics]).drop(columns=["confusion_matrix", "class_counts", "predicted_class_counts"]))

created_at = datetime.now(timezone.utc)
model_id = created_at.strftime("LSTM-OPTIONAL-%Y%m%dT%H%M%SZ")
run_dir = Path("artifacts/lstm") / model_id
run_dir.mkdir(parents=True, exist_ok=False)

predictions = pd.DataFrame({
    "schema_version": "1.0",
    "model_id": model_id,
    "ticker": prepared.symbol,
    "feature_date": pd.to_datetime(test_rows["feature_date"]).dt.strftime("%Y-%m-%d"),
    "target_date": pd.to_datetime(test_rows["target_date"]).dt.strftime("%Y-%m-%d"),
    "predicted_class": predicted_class,
    "probability_up": probability_up,
    "probability_down": 1.0 - probability_up,
    "actual_class": y_test,
})
predictions.to_csv(run_dir / "predictions.csv", index=False)
model.save(run_dir / "model.keras")
write_json(metrics, run_dir / "metrics.json")
write_json({
    "metadata_kind": "optional_lstm_notebook_experiment",
    "contract_status": "not_registered_model_artifact",
    "model_id": model_id,
    "algorithm": "lstm",
    "symbol": prepared.symbol,
    "random_seed": RANDOM_SEED,
    "sequence_length": SEQUENCE_LENGTH,
    "ordered_feature_columns": feature_columns,
    "dataset_sha256": prepared.dataset_sha256,
    "created_at": created_at.isoformat().replace("+00:00", "Z"),
    "status": "optional_experimental",
    "disclaimer": "Academic experiment; not financial advice.",
}, run_dir / "lstm_experimental_metadata.json")
zip_path = make_zip_from_directory(run_dir, run_dir.parent / f"{model_id}.zip")
print("Experimental metadata (not a registry manifest):", run_dir / "lstm_experimental_metadata.json")
print("Predictions (backtest contract):", run_dir / "predictions.csv")
print("ZIP:", zip_path)

## 6. Plot learning curves and optionally download

In [ ]:
import matplotlib.pyplot as plt

history_frame = pd.DataFrame(history.history)
history_frame[["loss", "val_loss"]].plot(title="LSTM loss")
plt.show()
history_frame[["accuracy", "val_accuracy"]].plot(title="LSTM accuracy")
plt.show()

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))